# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Show basic metadata about the dataset
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Discover available record sets, fields, and columns using @id
print("Available record sets (by @id and name):")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"@id: {rs.id} | name: {rs.name}")

# For demonstration, print first record set's fields & columns by @id
if len(record_sets) > 0:
    record_set = record_sets[0]
    print(f"\nFields for record set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | column @id: {field.column.id if field.column else None}")

    print(f"\nColumns for record set '{record_set.name}':")
    for col in record_set.columns:
        print(f"  Column @id: {col.id} | name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set, referencing them by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Preview column names and data for the first record set
if record_set_ids and record_set_ids[0] in dataframes:
    print("\nAvailable columns (by field @id) in the first record set dataframe:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Select a numeric field (by @id) and filter, normalize, group
import numpy as np
rs0_id = record_set_ids[0]
df = dataframes[rs0_id]

# Guess a likely numeric field (@id). You can update this if you know the schema.
possible_numeric_fields = [col for col in df.columns if 'coverage' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or df[col].dtype in [np.float64, np.int64]]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first found by heuristic
else:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0] if not df.empty else None
    
print(f"Selected numeric field for analysis: {numeric_field}")

if numeric_field and numeric_field in df.columns:
    # Drop NaNs for the numeric field
    filtered_df = df[df[numeric_field].notnull()]

    # Define a threshold (as an example, median value or 10 if suitable)
    threshold = np.percentile(filtered_df[numeric_field], 50)  # median
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median):")
    print(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a likely categorical field (e.g. with 'sample', 'condition', or 'group' in the name)
    possible_group_fields = [col for col in df.columns if 'sample' in col.lower() or 'condition' in col.lower() or 'group' in col.lower()]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No obvious categorical (group) field found for grouping in this dataset.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group, if available
    if possible_group_fields:
        group_field = possible_group_fields[0]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Through this notebook, we loaded and explored the protein mass spectrometry FAIR² dataset using the `mlcroissant` library. We inspected the available record sets and fields using their `@id`, extracted data into DataFrames, performed filtering and normalization on a numeric attribute, and visualized its distribution. For further data understanding, consult the dataset's schema and associated documentation (e.g., [FAIR² JSON-LD schema link](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)).